# Module 09: Interactive Concurrency — Threads, Multiprocessing & The GIL

Welcome to **Module 09**! Concurrency is one of the most misunderstood areas of Python. In this laboratory, you will explore:
1. **The Global Interpreter Lock (GIL):** Proving why Python threads do NOT accelerate CPU-bound math.
2. **I/O-Bound Workloads:** Showing where threading shines (parallel network/disk waits).
3. **Race Conditions on Shared State:** Reproducing lost increments and fixing them with `threading.Lock`.
4. **Multiprocessing:** Bypassing the GIL to achieve true parallel CPU core utilization.
5. **Process Communication:** Safely transferring data across process boundaries with `multiprocessing.Queue`.
6. **Interactive Challenge:** Building a thread-safe connection pool.


## 1. The Global Interpreter Lock (GIL) Benchmarking


In [ ]:
import threading
import time


def cpu_heavy_task(n: int) -> int:
    count = 0
    for i in range(n):
        count += i * i
    return count

WORKLOAD = 5_000_000

# 1. Sequential Execution
t0 = time.perf_counter()
cpu_heavy_task(WORKLOAD)
cpu_heavy_task(WORKLOAD)
seq_duration = time.perf_counter() - t0
print(f"Sequential (1 thread, 2 tasks): {seq_duration:.3f} s")

# 2. Multithreaded Execution (2 threads concurrently)
t0 = time.perf_counter()
t1 = threading.Thread(target=cpu_heavy_task, args=(WORKLOAD,))
t2 = threading.Thread(target=cpu_heavy_task, args=(WORKLOAD,))

t1.start()
t2.start()
t1.join()
t2.join()
threaded_duration = time.perf_counter() - t0
print(f"Multithreaded (2 threads, 2 tasks): {threaded_duration:.3f} s")
print(f"Ratio (Threaded / Sequential): {threaded_duration / seq_duration:.2f}x")
print("Notice: Threads took roughly equal or SLIGHTLY LONGER time due to GIL lock-switching overhead!")


## 2. Threading for I/O-Bound Workloads


In [ ]:
def simulated_io_fetch(endpoint_id: int):
    # Simulating network latency
    time.sleep(0.1)
    return f"Endpoint-{endpoint_id}: 200 OK"

t0 = time.perf_counter()
threads = [threading.Thread(target=simulated_io_fetch, args=(i,)) for i in range(10)]
for t in threads:
    t.start()
for t in threads:
    t.join()
io_duration = time.perf_counter() - t0
print(f"10 I/O fetches (0.1s each) finished in: {io_duration:.3f} s (Sequential would take 1.0s!)")


## 3. Race Conditions & Mutex Synchronization


In [ ]:
shared_counter = 0

def unsafe_worker(increments: int):
    global shared_counter
    for _ in range(increments):
        # Python bytecode interleaving can interrupt during read-modify-write!
        curr = shared_counter
        time.sleep(0.000001)
        shared_counter = curr + 1

t1 = threading.Thread(target=unsafe_worker, args=(500,))
t2 = threading.Thread(target=unsafe_worker, args=(500,))
t1.start()
t2.start()
t1.join()
t2.join()

print(f"Expected count: 1000, Actual count: {shared_counter}")
if shared_counter < 1000:
    print(f"[RACE DETECTED] Lost {1000 - shared_counter} increments due to unsynchronized memory access!")

# Fixing with threading.Lock:
lock = threading.Lock()
safe_counter = 0

def safe_worker(increments: int):
    global safe_counter
    for _ in range(increments):
        with lock:
            safe_counter += 1

t1 = threading.Thread(target=safe_worker, args=(500,))
t2 = threading.Thread(target=safe_worker, args=(500,))
t1.start()
t2.start()
t1.join()
t2.join()

print(f"Safe count with Mutex: {safe_counter} (100% correct)")
assert safe_counter == 1000


## 4. True CPU Parallelism with Multiprocessing


In [ ]:
from concurrent.futures import ProcessPoolExecutor

import nb_workers

# Multiprocessing spawns separate OS processes, each with its own CPython interpreter & memory!
# Functions submitted to ProcessPoolExecutor must be imported from an external module on spawn platforms (Windows/macOS)
WORKLOAD = 2_000_000

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(nb_workers.cpu_task, WORKLOAD) for _ in range(2)]
    results = [f.result() for f in futures]

multi_duration = time.perf_counter() - t0
print(f"Multiprocessing (2 separate processes): {multi_duration:.3f} s")
print("True parallel speedup achieved!")


## 5. Thread-Safe Producer-Consumer Queue


In [ ]:
import queue

work_queue = queue.Queue(maxsize=5)
produced = []
consumed = []

def producer():
    for item in range(10):
        work_queue.put(item)
        produced.append(item)
    work_queue.put(None)  # Sentinel to signal termination

def consumer():
    while True:
        item = work_queue.get()
        if item is None:
            work_queue.task_done()
            break
        consumed.append(item)
        work_queue.task_done()

prod_t = threading.Thread(target=producer)
cons_t = threading.Thread(target=consumer)
prod_t.start()
cons_t.start()
prod_t.join()
cons_t.join()

print("Produced items:", produced)
print("Consumed items:", consumed)
assert produced == consumed
print("[VERIFIED] All items safely transmitted across thread queue!")


## 6. Interactive Challenge: Thread-Safe Connection Pool


In [ ]:
# CHALLENGE: Implement a thread-safe connection pool with max capacity
# and acquire / release semantics using threading.Semaphore!

class ThreadSafeConnectionPool:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self._semaphore = threading.Semaphore(capacity)
        self._lock = threading.Lock()
        self._active_connections = 0

    def acquire(self) -> int:
        self._semaphore.acquire()
        with self._lock:
            self._active_connections += 1
            return self._active_connections

    def release(self):
        with self._lock:
            self._active_connections -= 1
        self._semaphore.release()

pool = ThreadSafeConnectionPool(capacity=3)

# Acquire all 3
c1 = pool.acquire()
c2 = pool.acquire()
c3 = pool.acquire()
print(f"Acquired 3 connections: {c1}, {c2}, {c3}")

# Releasing 1 allows next acquisition
pool.release()
c4 = pool.acquire()
print(f"Re-acquired connection after release: {c4}")
print("[ALL CONCURRENCY CHALLENGES PASSED]")
